# PlantLens — Colab test (Phase 0 + 1)
Run top to bottom. Needs: Colab GPU runtime, `HF_TOKEN` in Colab Secrets, photos in `MyDrive/plantlens/test_photos/`, skill folder in `MyDrive/plantlens/plantlens/`.

In [ ]:
!nvidia-smi
!pip -q install -U transformers accelerate bitsandbytes qwen-vl-utils pillow

In [ ]:
# ---- CONFIG (all settings live here) ----
import torch
from google.colab import drive, userdata
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/plantlens"          # photos, skill, results live here
CACHE_DIR = f"{BASE}/hf_cache"                     # model weights cached on Drive
PHOTO_DIR = f"{BASE}/test_photos"
SKILL_DIR = f"{BASE}/plantlens"

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {vram_gb:.1f} GB")

# ~16 GB (T4): 8B in 4-bit.  >=24 GB (L4/A100): 8B in bf16.  Step down to 4B / 2B if OOM.
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"             # verified on the HF hub
QUANT = "4bit" if vram_gb < 22 else "none"         # "4bit" | "8bit" | "none"
MAX_NEW_TOKENS = 700
TEMPERATURE = 0.3
MAX_IMAGE_PIXELS = 1024 * 28 * 28                  # caps visual tokens; saves memory

HF_TOKEN = userdata.get("HF_TOKEN")                # never printed
print(MODEL_ID, QUANT)

In [ ]:
# ---- Load model ----
import time, os
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

t0 = time.time()
kwargs = dict(cache_dir=CACHE_DIR, token=HF_TOKEN, device_map="auto")
if QUANT == "4bit":
    kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
elif QUANT == "8bit":
    kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
else:
    kwargs["dtype"] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **kwargs)
processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR, token=HF_TOKEN,
                                          max_pixels=MAX_IMAGE_PIXELS)
print(f"load time {time.time()-t0:.0f}s | GPU mem {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# ---- Generic chat helper ----
from PIL import Image
import glob

def chat(messages, max_new_tokens=None):
    """messages: list of {role, content:[{type:'image'|'text',...}]}"""
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens or MAX_NEW_TOKENS,
                             do_sample=TEMPERATURE > 0, temperature=TEMPERATURE or None)
    return processor.batch_decode(out[:, inputs["input_ids"].shape[1]:],
                                  skip_special_tokens=True)[0].strip()

photos = sorted(p for ext in ("jpg","jpeg","png","webp") for p in glob.glob(f"{PHOTO_DIR}/*.{ext}"))
assert photos, f"No photos in {PHOTO_DIR} — add some and rerun."
print(len(photos), "photos")

In [ ]:
# ---- Phase 0: smoke test ----
img = Image.open(photos[0]).convert("RGB")
display(img.resize((384, int(384*img.height/img.width))))
t0 = time.time()
msg = [{"role":"user","content":[{"type":"image","image":img},{"type":"text","text":"Describe what you see."}]}]
print(chat(msg, 300))
print(f"\n[{time.time()-t0:.1f}s | peak GPU mem {torch.cuda.max_memory_allocated()/1e9:.1f} GB]")

## Phase 1 — Skill wiring
System prompt = `SKILL.md` + `references/symptoms.md` (no examples yet).

In [ ]:
def read(p): return open(p, encoding="utf-8").read()
SYSTEM = read(f"{SKILL_DIR}/SKILL.md") + "\n\n# Symptom reference\n\n" + read(f"{SKILL_DIR}/references/symptoms.md")
print("system prompt chars:", len(SYSTEM))

def run_case(photo, user_text, history=None):
    """photo: path or None (follow-up turns). history: prior messages (without system)."""
    history = list(history or [])
    content = []
    if photo: content.append({"type":"image","image":Image.open(photo).convert("RGB")})
    content.append({"type":"text","text":user_text})
    history.append({"role":"user","content":content})
    msgs = [{"role":"system","content":[{"type":"text","text":SYSTEM}]}] + history
    reply = chat(msgs)
    history.append({"role":"assistant","content":[{"type":"text","text":reply}]})
    return reply, history

In [ ]:
# Pick 3 photos: healthy, clear problem, blurry — edit these filenames
PICKS = photos[:3]
for p in PICKS:
    print("="*80, "\n", os.path.basename(p), "\n", "="*80)
    t0 = time.time()
    reply, _ = run_case(p, "What's wrong with my plant?")
    print(reply, f"\n[{time.time()-t0:.1f}s]")